# 记忆整理方式1：消息裁剪

In [2]:
from langchain_core.messages import HumanMessage
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from typing import Any
from langchain_openai import ChatOpenAI

## 在调用模型之前使用的钩子函数
@before_model
def trim_message(state:AgentState, runtime:Runtime) -> dict[str, Any]|None:
    # 获取消息列表
    messages = state["messages"]
    # 如果消息列表长度小于3，不需要裁剪返回None
    if len(messages) <= 3:
        return None
    # 获取第一条消息,一定要保留的。
    first_msg = messages[0]
    # 如果有大于3的偶数条消息，则取最近的3条消息；如果有奇数条消息，则取最近的4条消息
    recent_msgs = messages[-3:] if len(messages) %2 == 0 else messages[-4:]
    new_msgs = [first_msg] + recent_msgs

    return  {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES), # 这一句的作用？
            *new_msgs,
        ]
    }


# 1.实例化模型
model = ChatOpenAI(
    model="gemma4:12b",
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)

# 2.创建agent对象
agent = create_agent(
    model=model,
    middleware=[trim_message],
    checkpointer=InMemorySaver(),
)

config = {"configurable":{"thread_id":"1"}}

agent.invoke({"messages":[HumanMessage("你好，我是老张")]},config=config)
agent.invoke({"messages":[HumanMessage("从现在起，你叫小张")]},config=config)
agent.invoke({"messages":[HumanMessage("今天天气不错")]},config=config)
resp = agent.invoke({"messages":[HumanMessage("我是谁，你是谁？")]},config=config)

for msg in resp["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好，我是老张
================================== Ai Message ==================================

没问题，没问题！从现在起，我就叫**小张**了。

那以后您就叫我小张吧，老张。请问今天有什么事需要小张帮您的？
================================ Human Message =================================

今天天气不错
================================== Ai Message ==================================

确实，好的天气最能让人心情愉快了！

既然天气这么好，老张您今天打算怎么安排呢？是打算出门去逛逛、活动活动，还是在家里悠闲地享受这份好天气？

如果您有任何想聊的，或者需要小张帮您查点资料、规划个路线，随时吩咐我！
================================ Human Message =================================

我是谁，你是谁？
================================== Ai Message ==================================

您是**老张**，刚才您才刚跟我打过招呼呢！

我是**小张**，也就是您的AI助手。

如果从技术层面来说，我是一个由 Google 训练的大型语言模型。

所以，现在的对话就是**老张**和**小张**在聊天。您还有什么想聊的，或者需要小张帮您忙的吗？
